# CARec encoder and dimension study

This notebook will compare general-purpose and instruction-based text encoders as item representations in CARec. We plan to keep the dataset split and recommender setup fixed, then measure warm- and cold-item Recall@10 and NDCG@10 at shared embedding sizes. First, we need to reproduce a baseline.

## References

- [CARec paper, CIKM 2024](https://doi.org/10.1145/3627673.3679535)
- [Official CARec repository](https://github.com/ChenMetanoia/CARec)
- [CARec runner](https://github.com/ChenMetanoia/CARec/blob/main/run_collabcontex.py)
- [CARec setup instructions](https://github.com/ChenMetanoia/CARec#readme)

## Planned sections

1. Set the experiment configuration and random seeds.
2. Load item metadata and the fixed interaction split.
3. Encode item text with the selected general-purpose or instruction-based model.
4. Cache embeddings and reduce them to the target dimensions using the agreed method.
5. Load the item embeddings into CARec and train the recommender.
6. Evaluate warm and cold items, then compare ranking quality, runtime, and storage.

Raw Electronics JSON files are in `data/`. The pinned CARec checkout is in `external/CARec`. Select the project `.venv` interpreter or the NI-PML benchmarks kernel. Run `bash scripts/setup.sh` from the repository root to restore the environment.

## Imports and device

Use Apple's MPS backend when PyTorch reports it available. The fallbacks let the notebook run on another machine too.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch

mps_backend = getattr(torch.backends, "mps", None)
if mps_backend is not None and mps_backend.is_available():
    DEVICE = torch.device("mps")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")

print(f"Using device: {DEVICE}")

Using device: mps


## Data preparation: paths and preview

Start here to inspect the raw data. These cells run independently of the device and CARec imports. Find the project root from either the repository or `src/`, check both files, and read only their first five lines. Each JSONL line contains one JSON record.

This is a preview of the file beginnings, not a random sample or an evaluation dataset. It does not establish data quality across the full files.

In [1]:
from pathlib import Path

PROJECT_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "requirements.txt").is_file()
)
DATA_DIR = PROJECT_ROOT / "data"
REVIEWS_PATH = DATA_DIR / "Electronics.json"
METADATA_PATH = DATA_DIR / "meta_Electronics.json"
PREVIEW_ROWS = 5

for path in (REVIEWS_PATH, METADATA_PATH):
    if not path.is_file():
        raise FileNotFoundError(f"Missing data file: {path}")
    print(f"{path.relative_to(PROJECT_ROOT)}: {path.stat().st_size / 1e9:.2f} GB")

data/Electronics.json: 11.51 GB
data/meta_Electronics.json: 11.00 GB


### Data preview

In [2]:
import json
from itertools import islice

import pandas as pd
from IPython.display import display


def read_jsonl_preview(path, limit=5):
    if limit < 1:
        raise ValueError("Preview limit must be positive.")
    records = []
    with path.open(encoding="utf-8") as source:
        for line_number, line in enumerate(islice(source, limit), start=1):
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as error:
                raise ValueError(f"Invalid JSON in {path}, line {line_number}") from error
    return records

In [3]:
review_preview = read_jsonl_preview(REVIEWS_PATH, PREVIEW_ROWS)
print(f"Reviews: first {len(review_preview)} records")
display(pd.DataFrame(review_preview))

Reviews: first 5 records


,overall,verified,reviewTime,reviewerID,asin,style,reviewerName,reviewText,summary,unixReviewTime,vote
0,5.0,True,"07 17, 2002",A1N070NS9CJQ2I,0060009810,{'Format:': ' Hardcover'},Teri Adams,This was the first time I read Garcia-Aguilera...,Hit The Spot!,1026864000,NaN
1,5.0,False,"07 6, 2002",A3P0KRKOBQK1KN,0060009810,{'Format:': ' Hardcover'},Willa C.,"As with all of Ms. Garcia-Aguilera's books, I ...",one hot summer is HOT HOT HOT!,1025913600,NaN
2,5.0,False,"07 3, 2002",A192HO2ICJ75VU,0060009810,{'Format:': ' Hardcover'},Kit,I've not read any of Ms Aguilera's works befor...,One Hot Summer,1025654400,2
3,4.0,False,"06 30, 2002",A2T278FKFL3BLT,0060009810,{'Format:': ' Hardcover'},Andres,This romance novel is right up there with the ...,I love this book!,1025395200,3
4,5.0,False,"06 28, 2002",A2ZUXVTW8RXBXW,0060009810,{'Format:': ' Hardcover'},John,Carolina Garcia Aguilera has done it again. S...,One Hot Book,1025222400,NaN


In [4]:
metadata_preview = read_jsonl_preview(METADATA_PATH, PREVIEW_ROWS)
print(f"Metadata: first {len(metadata_preview)} records")
display(pd.DataFrame(metadata_preview))

Metadata: first 5 records


,category,tech1,description,fit,title,also_buy,tech2,brand,feature,rank,also_view,main_cat,similar_item,date,price,asin,imageURL,imageURLHighRes
0,"[Electronics, Camera &amp; Photo, Video Survei...",,[The following camera brands and models have b...,,Genuine Geovision 1 Channel 3rd Party NVR IP S...,[],,GeoVision,"[Genuine Geovision 1 Channel NVR IP Software, ...","[>#3,092 in Tools &amp; Home Improvement &gt; ...",[],Camera &amp; Photo,,"January 28, 2014",$65.00,0011300000,[https://images-na.ssl-images-amazon.com/image...,[https://images-na.ssl-images-amazon.com/image...
1,"[Electronics, Camera &amp; Photo]",,[This second edition of the Handbook of Astron...,,"Books ""Handbook of Astronomical Image Processi...",[0999470906],,33 Books Co.,[Detailed chapters cover these fundamental top...,"[>#55,933 in Camera &amp; Photo (See Top 100 i...","[0943396670, 1138055360, 0999470906]",Camera &amp; Photo,,"June 17, 2003",,0043396828,[https://images-na.ssl-images-amazon.com/image...,[https://images-na.ssl-images-amazon.com/image...
2,"[Electronics, eBook Readers &amp; Accessories,...",,[A zesty tale. (Publishers Weekly)<br /><br />...,,One Hot Summer,"[0425167798, 039914157X]",,Visit Amazon's Carolina Garcia Aguilera Page,[],"3,105,177 in Books (",[],Books,,,$11.49,0060009810,[],[]
3,"[Electronics, eBook Readers & Accessories, eBo...",,[],,Hurray for Hattie Rabbit: Story and pictures (...,"[0060219521, 0060219580, 0060219394]",,Visit Amazon's Dick Gackenbach Page,[],"2,024,298 in Books (","[0060219521, 0060219475, 0060219394]",Books,,,.a-section.a-spacing-mini{margin-bottom:6px!im...,0060219602,[],[]
4,"[Electronics, eBook Readers & Accessories, eBo...",,[&#8220;sex.lies.murder.fame. is brillllli&#82...,,sex.lies.murder.fame.: A Novel,[],,Visit Amazon's Lolita Files Page,[],"3,778,828 in Books (",[],Books,,,$13.95,0060786817,[],[]


## CARec imports

Run the data paths cell above first. This loads the upstream model and data/evaluation helpers without starting preprocessing or training. The wrapper checks the upstream commit and refuses local source changes.

In [ ]:
import sys

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from benchmarks.carec import load_carec
from recbole.config import Config
from recbole.trainer import Trainer

carec = load_carec()
CollabContex = carec.CollabContex
ColdEvaluator = carec.ColdEvaluator
data_preparation = carec.data_preparation

print(f"CARec: {carec.commit}")